In [ ]:
import pandas as pd

appointments = pd.read_csv(
    "https://raw.githubusercontent.com/"
    "jyotikeshri99999/Capstone-Project/main/appointments_clean.csv"
)
clinics = pd.read_csv(
    "https://raw.githubusercontent.com/"
    "jyotikeshri99999/Capstone-Project/main/clinics_enriched.csv"
)

clinic_check = (
    appointments.groupby("clinic_id")
    .agg(
        appointments=("appointment_id", "count"),
        no_shows=("no_show", "sum")
    )
    .merge(clinics[["clinic_id", "clinic_name", "city"]], on="clinic_id")
)

clinic_check["no_show_pct"] = (
    100 * clinic_check["no_shows"] / clinic_check["appointments"]
).round(1)

print(clinic_check.sort_values("no_show_pct", ascending=False).to_string(index=False))

clinic_id  appointments  no_shows              clinic_name    city  no_show_pct
      C08            90        30   Gomti Nagar Care Point Lucknow         33.3
      C02           143        39     Hadapsar Care Centre    Pune         27.3
      C04           142        38     Malviya Nagar Clinic  Jaipur         26.8
      C01           160        36    Kothrud Family Clinic    Pune         22.5
      C07           101        21       Vijay Nagar Clinic  Indore         20.8
      C06           142        23 Edappally Medical Centre   Kochi         16.2
      C03            82        13   Sitabuldi Health Point  Nagpur         15.9
      C05           130        18    Vaishali Nagar Clinic  Jaipur         13.8


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

url = (
    "https://raw.githubusercontent.com/"
    "jyotikeshri99999/Capstone-Project/main/appointments_clean.csv"
)
appointments = pd.read_csv(url)

# Keep IDs and target aside; neither is used as a feature.
appointment_ids = appointments["appointment_id"]
y = appointments["no_show"]

# Start with the six specified numeric/binary features.
X = appointments[
    [
        "age",
        "distance_km",
        "fee_inr",
        "lead_days",
        "prior_no_shows",
        "reminder_sent"
    ]
].copy()

# Convert reminder values to 1 and 0.
X["reminder_sent"] = X["reminder_sent"].map({"Yes": 1, "No": 0})

# Add one-hot columns for booking channel and visit type.
X["booking_channel"] = appointments["booking_channel"]
X["visit_type"] = appointments["visit_type"]
X = pd.get_dummies(
    X,
    columns=["booking_channel", "visit_type"],
    dtype=int
)

# Keep the exact split required by the assignment.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Feature columns:", list(X.columns))
print("Number of features:", X.shape[1])
print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("No-shows in test set:", int(y_test.sum()))

# Baseline: always predict Attended (0).
baseline_accuracy = (y_test == 0).mean()
print("Always-Attended baseline accuracy:", baseline_accuracy)
print("Baseline correct predictions:", int((y_test == 0).sum()), "of", len(y_test))

Feature columns: ['age', 'distance_km', 'fee_inr', 'lead_days', 'prior_no_shows', 'reminder_sent', 'booking_channel_App', 'booking_channel_Phone', 'booking_channel_Walk-In', 'visit_type_Consultation', 'visit_type_Follow-up']
Number of features: 11
Training rows: 792
Test rows: 198
No-shows in test set: 49
Always-Attended baseline accuracy: 0.7525252525252525
Baseline correct predictions: 149 of 198


In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# Scale the features using training data only.
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# Fit logistic regression and predict on the test set.
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_s, y_train)

y_pred = lr.predict(X_test_s)
y_prob = lr.predict_proba(X_test_s)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

# Show coefficients in the feature-column order.
coefficients = pd.Series(lr.coef_[0], index=X.columns)
print("\nCoefficients:")
print(coefficients.sort_values())

# Reminder coefficient interpretation.
reminder_coef = coefficients["reminder_sent"]
reminder_index = list(X.columns).index("reminder_sent")
reminder_scale = scaler.scale_[reminder_index]

print("\nReminder coefficient:", reminder_coef)
print("Odds multiplier per 1 SD:", np.exp(reminder_coef))
print("Reminder feature SD:", reminder_scale)
print("Odds multiplier for No → Yes:", np.exp(reminder_coef / reminder_scale))

Accuracy: 0.7878787878787878
Confusion matrix:
 [[139  10]
 [ 32  17]]
Precision: 0.6296296296296297
Recall: 0.3469387755102041
F1: 0.4473684210526316
ROC-AUC: 0.788933022873579

Coefficients:
reminder_sent             -0.580669
age                       -0.253880
booking_channel_Walk-In   -0.186364
booking_channel_App       -0.046804
visit_type_Follow-up      -0.015658
visit_type_Consultation    0.015658
booking_channel_Phone      0.215881
distance_km                0.253448
fee_inr                    0.452761
lead_days                  0.614471
prior_no_shows             0.649612
dtype: float64

Reminder coefficient: -0.5806686033659922
Odds multiplier per 1 SD: 0.559524141750829
Reminder feature SD: 0.4949752621616814
Odds multiplier for No → Yes: 0.3093980903124433


In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Full-depth tree.
tree_full = DecisionTreeClassifier(random_state=42)
tree_full.fit(X_train, y_train)

full_train_accuracy = accuracy_score(y_train, tree_full.predict(X_train))
full_test_accuracy = accuracy_score(y_test, tree_full.predict(X_test))

print("Full-depth tree")
print("Depth:", tree_full.get_depth())
print("Training accuracy:", full_train_accuracy)
print("Test accuracy:", full_test_accuracy)

# Tree limited to depth 3.
tree_depth3 = DecisionTreeClassifier(max_depth=3, random_state=42)
tree_depth3.fit(X_train, y_train)

depth3_train_accuracy = accuracy_score(y_train, tree_depth3.predict(X_train))
depth3_test_accuracy = accuracy_score(y_test, tree_depth3.predict(X_test))

print("\nDepth-3 tree")
print("Depth:", tree_depth3.get_depth())
print("Training accuracy:", depth3_train_accuracy)
print("Test accuracy:", depth3_test_accuracy)

Full-depth tree
Depth: 15
Training accuracy: 1.0
Test accuracy: 0.6717171717171717

Depth-3 tree
Depth: 3
Training accuracy: 0.821969696969697
Test accuracy: 0.7777777777777778


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, recall_score, roc_auc_score
import pandas as pd

# Fit the random forest on the unscaled training features.
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

rf_train_accuracy = accuracy_score(y_train, rf.predict(X_train))
rf_pred = rf.predict(X_test)
rf_prob = rf.predict_proba(X_test)[:, 1]

rf_accuracy = accuracy_score(y_test, rf_pred)
rf_recall = recall_score(y_test, rf_pred)
rf_auc = roc_auc_score(y_test, rf_prob)

print("Random forest training accuracy:", rf_train_accuracy)
print("Random forest test accuracy:", rf_accuracy)
print("Random forest test recall:", rf_recall)
print("Random forest test ROC-AUC:", rf_auc)

# List the three features with the largest importance scores.
importance = pd.Series(rf.feature_importances_, index=X.columns)
print("\nTop three feature importances:")
print(importance.sort_values(ascending=False).head(3))

# Compare the models on the test set.
comparison = pd.DataFrame([
    {
        "model": "Logistic regression",
        "accuracy": accuracy_score(y_test, lr.predict(X_test_s)),
        "recall": recall_score(y_test, lr.predict(X_test_s)),
        "roc_auc": roc_auc_score(y_test, lr.predict_proba(X_test_s)[:, 1])
    },
    {
        "model": "Depth-3 decision tree",
        "accuracy": accuracy_score(y_test, tree_depth3.predict(X_test)),
        "recall": recall_score(y_test, tree_depth3.predict(X_test)),
        "roc_auc": roc_auc_score(
            y_test, tree_depth3.predict_proba(X_test)[:, 1]
        )
    },
    {
        "model": "Random forest",
        "accuracy": rf_accuracy,
        "recall": rf_recall,
        "roc_auc": rf_auc
    }
])

print("\nTest-set comparison:")
print(comparison.round(3).to_string(index=False))

Random forest training accuracy: 1.0
Random forest test accuracy: 0.7474747474747475
Random forest test recall: 0.30612244897959184
Random forest test ROC-AUC: 0.7057252431173813

Top three feature importances:
lead_days      0.219905
age            0.196788
distance_km    0.194739
dtype: float64

Test-set comparison:
                model  accuracy  recall  roc_auc
  Logistic regression     0.788   0.347    0.789
Depth-3 decision tree     0.778   0.224    0.700
        Random forest     0.747   0.306    0.706


In [ ]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, cross_val_score, cross_val_predict
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Logistic regression pipeline: scaling is re-fitted inside each CV fold.
lr_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

tree3_cv_model = DecisionTreeClassifier(max_depth=3, random_state=42)
rf_cv_model = RandomForestClassifier(n_estimators=100, random_state=42)

# Tune the forest using training data only.
grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    {
        "n_estimators": [100, 200],
        "max_depth": [3, 5, 8]
    },
    cv=5,
    scoring="roc_auc"
)
grid.fit(X_train, y_train)

print("Tuned forest best parameters:", grid.best_params_)
print("Tuned forest best CV ROC-AUC:", grid.best_score_)

# Compare mean CV ROC-AUC scores on training data only.
lr_cv_auc = cross_val_score(
    lr_pipe, X_train, y_train, cv=5, scoring="roc_auc"
).mean()

tree3_cv_auc = cross_val_score(
    tree3_cv_model, X_train, y_train, cv=5, scoring="roc_auc"
).mean()

rf_cv_auc = cross_val_score(
    rf_cv_model, X_train, y_train, cv=5, scoring="roc_auc"
).mean()

cv_scores = {
    "Logistic regression": lr_cv_auc,
    "Depth-3 tree": tree3_cv_auc,
    "Random forest": rf_cv_auc,
    "Tuned random forest": grid.best_score_
}

print("\nTraining cross-validation ROC-AUC:")
for model_name, score in cv_scores.items():
    print(f"{model_name}: {score:.3f}")

# Select the model with the highest training CV ROC-AUC.
if max(cv_scores, key=cv_scores.get) == "Logistic regression":
    winner_name = "Logistic regression"
    winner = lr_pipe
elif max(cv_scores, key=cv_scores.get) == "Depth-3 tree":
    winner_name = "Depth-3 tree"
    winner = tree3_cv_model
elif max(cv_scores, key=cv_scores.get) == "Random forest":
    winner_name = "Random forest"
    winner = rf_cv_model
else:
    winner_name = "Tuned random forest"
    winner = grid.best_estimator_

print("\nSelected model:", winner_name)
print("Selected CV ROC-AUC:", max(cv_scores.values()))

# Out-of-fold training probabilities; use these to choose a threshold.
oof_prob = cross_val_predict(
    winner,
    X_train,
    y_train,
    cv=5,
    method="predict_proba"
)[:, 1]

threshold_results = []

for threshold in [0.5, 0.35]:
    oof_flag = (oof_prob >= threshold).astype(int)
    threshold_results.append({
        "threshold": threshold,
        "precision": precision_score(y_train, oof_flag, zero_division=0),
        "recall": recall_score(y_train, oof_flag, zero_division=0),
        "f1": f1_score(y_train, oof_flag, zero_division=0),
        "flagged": int(oof_flag.sum())
    })

print("\nOut-of-fold training results:")
print(pd.DataFrame(threshold_results).round(3).to_string(index=False))

# Choose the threshold with the higher out-of-fold F1.
chosen_threshold = max(threshold_results, key=lambda row: row["f1"])["threshold"]
print("\nChosen threshold:", chosen_threshold)

# Fit the selected model on all training rows, then evaluate on the test set once.
winner.fit(X_train, y_train)
test_prob = winner.predict_proba(X_test)[:, 1]
test_flag = (test_prob >= chosen_threshold).astype(int)

print("\nOne test-set evaluation:")
print("Precision:", precision_score(y_test, test_flag, zero_division=0))
print("Recall:", recall_score(y_test, test_flag, zero_division=0))
print("Flagged:", int(test_flag.sum()))
print("ROC-AUC:", roc_auc_score(y_test, test_prob))

# Save a risk score for every test row.
test_risk_scores = pd.DataFrame({
    "appointment_id": appointments.loc[X_test.index, "appointment_id"].to_numpy(),
    "no_show": y_test.to_numpy(),
    "risk_prob": test_prob,
    "flag": test_flag
})
test_risk_scores.to_csv("test_risk_scores.csv", index=False)

print("\nSaved test_risk_scores.csv with", len(test_risk_scores), "rows")

Tuned forest best parameters: {'max_depth': 3, 'n_estimators': 200}
Tuned forest best CV ROC-AUC: 0.7673242251739405

Training cross-validation ROC-AUC:
Logistic regression: 0.774
Depth-3 tree: 0.678
Random forest: 0.747
Tuned random forest: 0.767

Selected model: Logistic regression
Selected CV ROC-AUC: 0.7737139899948249

Out-of-fold training results:
 threshold  precision  recall    f1  flagged
      0.50      0.575   0.249 0.347       73
      0.35      0.513   0.467 0.489      154

Chosen threshold: 0.35

One test-set evaluation:
Precision: 0.5714285714285714
Recall: 0.5714285714285714
Flagged: 49
ROC-AUC: 0.788933022873579

Saved test_risk_scores.csv with 198 rows
